In [1]:
from database.MongoDBClient import MongoDBClient
from database.Recipe import Recipe
from transformers import AutoTokenizer
from model.model import Chefformer
from config.ModelSettings import ModelSettings
import torch

/Users/michalekb/Documents/GitHub_Personal/chefformer_repo/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/michalekb/Documents/GitHub_Personal/chefformer_repo/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ModelSettings()

ModelSettings(name='Chefformer', vocab_size=50257, embedding_size=768, max_context_length=512, num_attn_heads=1)

In [3]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
chefformer = Chefformer(tokenizer)


In [4]:
s = """Title: Italian Cream Cake

Ingredients:
- Buttermilk (1 cup)
- Baking soda (1 teaspoon)
- Butter (1/2 cup)
- Shortening (1/2 cup)
- White sugar (2 cups)
- Eggs (5)
- Vanilla extract (1 teaspoon)
- Flaked coconut (1 cup)
- Baking powder (1 teaspoon)
- All-purpose flour (2 cups)
- Cream cheese (8 ounces)
- Butter (1/2 cup)
- Vanilla extract (1 teaspoon)
- Confectioners' sugar (4 cups)
- Light cream (2 tablespoons)
- Chopped walnuts (1/2 cup)
- Sweetened flaked coconut (1 cup)
...

Instructions: Finely grate the zest from 4 of the lemons, then pulse in a mini food processor with 3 tablespoons salt. Squeeze the juice from 2 of the zested lemons into a large bowl; add the lemongrass, olive oil, Sriracha, garlic, fish sauce, brown sugar and 1 teaspoon of the lemon salt. Thinly slice the 2 remaining lemons and add to the bowl.Using kitchen shears, cut through the shrimp shells along the outer curve, leaving the shells on. Remove the veins and rinse the shrimp. Add to the bowl with the lemongrass marinade and toss; cover and refrigerate for 1 hour. Meanwhile, soak 12 to 15 wooden skewers in water.Preheat a grill to medium high. Thread the shrimp onto the skewers. Brush the grill with vegetable oil, then add the lemon slices and grill until they begin to char, turning once, about 2 minutes; transfer to a platter. Stir the mint into the remaining marinade. Add the shrimp skewers to the grill and cook, brushing with the marinade-mint mixture, until the shells begin to char, 2 to 3 minutes per side. Transfer to the platter and sprinkle with the lemon salt to taste."""


s = 'hey how are you'

s = ['hey how', 'are you doing today']

In [5]:
x = tokenizer(s, return_tensors='pt', padding=True, truncation=True, max_length=512)
x['input_ids']

tensor([[20342,   703, 50256, 50256],
        [  533,   345,  1804,  1909]])

In [6]:
embeddings = chefformer.get_embeddings(s)
print(embeddings)
print(embeddings.shape)

tensor([[[-2.3038, -1.0551, -2.3138,  ..., -1.0959,  0.1786, -0.8018],
         [-0.8612,  1.2931, -2.5925,  ..., -0.0275, -0.2104, -0.3296],
         [-2.2799, -0.5765,  0.3969,  ..., -2.1746, -1.2232,  0.7983],
         [-2.2207, -0.3249,  0.9692,  ...,  0.0584, -1.9215,  3.2893]],

        [[-1.4935, -1.7338,  0.1332,  ..., -0.6021,  2.2289, -1.7674],
         [ 0.0399,  3.1322, -3.6858,  ..., -0.1368,  0.5330, -0.7073],
         [-0.4629, -2.2596, -0.7937,  ..., -2.0368, -1.1313, -0.8699],
         [-1.7776,  0.0843, -0.6049,  ...,  0.9925, -2.2405,  2.7538]]],
       grad_fn=<AddBackward0>)
torch.Size([2, 4, 768])


In [7]:
# Test parts of attention

def scaled_dot_product_attention(query, key, value):
    batch_size, seq_length, d_attn_head = query.shape[0], query.shape[1], query.shape[-1]
    attenion_weights = torch.bmm(query, torch.transpose(key, 1, 2)) # Q * K.T... (batch_size, seq_len, seq_len)
    attenion_weights = attenion_weights / torch.sqrt(torch.tensor(d_attn_head, dtype=torch.float32)) # Scale the scores

    mask = torch.tril(torch.ones(batch_size, seq_length, seq_length, dtype=torch.int)) # Gather attention matrix... (batch_size, seq_len, seq_len)
    attenion_weights[mask == 0] = float('-inf') # Set scores to -inf where tokens are masked

    attenion_weights = torch.softmax(attenion_weights, dim=-1) # Take softmax along rows (each token's row should sum to 1)
    return torch.bmm(attenion_weights, value)


example_embeddings = torch.tensor([[[0.5, 0.5, 0.5],
                           [-0.1, -0.1, -0.1]],

                           [[0.3, 0.3, 0.3],
                            [1.0, 1.0, 1.0]]])

print(scaled_dot_product_attention(example_embeddings, example_embeddings, example_embeddings))
print()


attn_output = chefformer.test_attention_head(['hey how are you', 'today'])
print(attn_output)
print(attn_output.shape)

tensor([[[0.5000, 0.5000, 0.5000],
         [0.1844, 0.1844, 0.1844]],

        [[0.3000, 0.3000, 0.3000],
         [0.8395, 0.8395, 0.8395]]])

tensor([[[-0.2802,  0.2090,  0.8878,  ..., -1.0918, -1.9770,  1.3653],
         [-0.5661,  0.1607,  0.8090,  ..., -0.6702, -1.6019,  0.8827],
         [-0.3458,  0.2013,  0.6772,  ..., -0.8304, -1.5118,  0.8067],
         [-0.1175,  0.0205,  0.2125,  ..., -0.3167, -0.6523, -0.2568]],

        [[-0.6191,  0.8060, -0.3878,  ..., -0.7603, -1.5970,  1.2944],
         [-0.6473,  0.6397, -0.2533,  ..., -0.5177, -1.3339,  1.1132],
         [-0.4968,  0.3582, -0.0807,  ..., -0.2926, -1.0964,  0.9148],
         [-0.3539, -0.2242,  0.1901,  ...,  0.3070, -0.4270,  0.4693]]],
       grad_fn=<BmmBackward0>)
torch.Size([2, 4, 768])
